# GNN node-encoding runs (rwpe16; four operators, configurations 4 and 5)

Runs on a Kaggle GPU notebook: the four operators on configurations 4 and 5, the fixed hyperparameters (`conf/hyperparameters/gnn/fixed.yaml`), with 16 random-walk columns (`rwpe16`) appended to the 6 entity one-hots of every node. Eight runs, about 8 hours.
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the datasets `sandrokhizanishvili/hi-small-gnn` and `sandrokhizanishvili/hi-small-rwpe`.
- Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/<model>/<run>_enc-rwpe16/seed-42/`.

In [1]:
MODELS = 'gin,pna,gat,transformer'
FEATURE_CONFIGS = 'c4,c5'
SEED = 42

In [2]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

Found existing installation: torch 2.11.0+cu130
Uninstalling torch-2.11.0+cu130:
  Successfully uninstalled torch-2.11.0+cu130
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 209.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 425.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 457.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 355.7 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 395.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 440.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [3]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml

Cloning into '/kaggle/working/gnn_for_aml'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 175 (delta 72), reused 147 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 335.84 KiB | 18.66 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/kaggle/working/gnn_for_aml


In [4]:
# One data folder: the graphs and rwpe/ linked from the two datasets (in /tmp, so the links stay out of the Output tab).
INPUT = '/kaggle/input/datasets/sandrokhizanishvili'
!mkdir -p /tmp/data
!ln -sf {INPUT}/hi-small-gnn/train_graph.pt {INPUT}/hi-small-gnn/val_graph.pt {INPUT}/hi-small-gnn/test_graph.pt {INPUT}/hi-small-gnn/feature_meta.json /tmp/data/
!ln -sfn {INPUT}/hi-small-rwpe /tmp/data/rwpe
!ls -lL /tmp/data /tmp/data/rwpe

/tmp/data:
total 4284450
-rw-rw-rw- 1 root root       4717 Sep  1 19:04 feature_meta.json
drwxrwxrwx 1 root root          0 Nov 10 20:26 gfp_variants
drwxrwxrwx 1 root root          0 Nov 10 20:26 rwpe
-rw-rw-rw- 1 root root 1824938361 Sep  1 19:05 test_graph.pt
-rw-rw-rw- 1 root root 1099908869 Sep  1 19:04 train_graph.pt
-rw-rw-rw- 1 root root 1462423405 Sep  1 19:05 val_graph.pt

/tmp/data/gfp_variants:
total 1269311
-rw-rw-rw- 1 root root       1475 Oct  5 18:14 tuned_cols.json
-rw-rw-rw- 1 root root 1299772800 Oct  5 18:14 tuned.npy

/tmp/data/rwpe:
total 144875
-rw-rw-rw- 1 root root 32966099 Oct  4 12:25 rwpe_k16_test.pt
-rw-rw-rw- 1 root root 32966106 Oct  4 12:25 rwpe_k16_train.pt
-rw-rw-rw- 1 root root 32966092 Oct  4 12:25 rwpe_k16_val.pt
-rw-rw-rw- 1 root root 16483852 Oct  4 12:25 rwpe_k8_test.pt
-rw-rw-rw- 1 root root 16483859 Oct  4 12:25 rwpe_k8_train.pt
-rw-rw-rw- 1 root root 16483845 Oct  4 12:25 rwpe_k8_val.pt


In [5]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

python 3.13.16 | torch 2.8.0+cu126 | CUDA build 12.6 | device cuda
  torch_geometric 2.8.1
  torch_scatter 2.1.2+pt28cu126
  torch_sparse 0.6.18+pt28cu126
  lightgbm 4.6.0
  hydra-core 1.3.2
  hydra-submitit-launcher not installed (only needed on Cineca)
  numpy 2.1.3
  scipy 1.16.3
  pandas 2.3.3
  scikit-learn 1.6.1
  matplotlib 3.10.0
  psutil 5.9.5
/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
  gin         params   54,083 | loss 0.7145
  pna         params  176,833 | loss 0.7128
  gat         params   53,953 | loss 0.8115
  transformer params   70,337 | loss 0.7536
OK


In [6]:
# The eight runs, one after another. A finished run is skipped, so the cell can be rerun after an interruption.
!python run.py -m model={MODELS} feature_config={FEATURE_CONFIGS} node_encoding=rwpe16 seed={SEED} machine=kaggle paths.data=/tmp/data

[2026-11-10 20:30:14,196][HYDRA] Launching 8 jobs locally
[2026-11-10 20:30:14,197][HYDRA] 	#0 : model=gin feature_config=c4 node_encoding=rwpe16 seed=42 machine=kaggle paths.data=/tmp/data
=== train gin_mp-full_readout-full_dir-in_enc-rwpe16 seed 42 -> /kaggle/working/outputs/train/gin/gin_mp-full_readout-full_dir-in_enc-rwpe16/seed-42
mode: train
seed: 42
trial: null
node_encoding: rwpe16
gfp_sheet: default
hyperparameters:
  for: gnn
  hidden: 128
  num_layers: 2
  dropout: 0.3
  norm: none
  residual: identity
  ffn: false
  num_neighbors:
  - 100
  - 100
  batch_size: 8192
  epochs: 20
  lr: 0.001
  weight_decay: 1.0e-05
  pos_weight: 8.0
  self_loops: true
  readout_layers: 2
  readout_width: 1.0
  lr_schedule: cosine
  num_workers: 0
  select_metric: pr_auc
model:
  name: gin
  kind: gnn
feature_config:
  id: 4
  mp: full
  readout: full
machine: kaggle
paths:
  data: /tmp/data
  outputs: /kaggle/working/outputs

edge_attr width 81 (base 20 + gfp 61, sheet default)
train nodes=5

In [7]:
# The metrics of every finished run, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob('/kaggle/working/outputs/train/*/*_enc-rwpe16/seed-*/results.json'))]
cols = ['run', 'best_epoch', 'threshold', 'params', 'invariant_params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

,run,best_epoch,threshold,params,invariant_params,train_f1,train_precision,train_recall,train_pr_auc,val_f1,val_precision,val_recall,val_pr_auc,test_f1,test_precision,test_recall,test_pr_auc
0,gat_mp-base_readout-full_dir-in_enc-rwpe16,19,0.5603,118017,67585,0.4898,0.6359,0.3983,0.4363,0.5691,0.7764,0.4492,0.5144,0.5085,0.6778,0.4068,0.4639
1,gat_mp-full_readout-full_dir-in_enc-rwpe16,15,0.5855,133633,67585,0.4941,0.5980,0.4210,0.4581,0.5676,0.7527,0.4556,0.5160,0.4997,0.6247,0.4164,0.4630
2,gin_mp-base_readout-full_dir-in_enc-rwpe16,16,0.6713,118275,67587,0.5760,0.8151,0.4454,0.5372,0.6133,0.8680,0.4741,0.5588,0.5348,0.7256,0.4234,0.4859
3,gin_mp-full_readout-full_dir-in_enc-rwpe16,11,0.6614,133891,67587,0.5737,0.7559,0.4623,0.5560,0.5967,0.8579,0.4575,0.5461,0.5120,0.7786,0.3815,0.4735
4,pna_mp-base_readout-full_dir-in_enc-rwpe16,13,0.8400,609537,427265,0.5822,0.7950,0.4593,0.5458,0.6496,0.8470,0.5268,0.6121,0.6142,0.7906,0.5022,0.5710
5,pna_mp-full_readout-full_dir-in_enc-rwpe16,16,0.6894,625153,427265,0.6083,0.8415,0.4763,0.6047,0.6386,0.8694,0.5046,0.5965,0.6132,0.8242,0.4882,0.5712
6,transformer_mp-base_readout-full_dir-in_enc-rw...,18,0.5979,183553,133121,0.5742,0.7505,0.4650,0.5521,0.6229,0.8041,0.5083,0.5734,0.5944,0.7575,0.4891,0.5591
7,transformer_mp-full_readout-full_dir-in_enc-rw...,11,0.7586,199169,133121,0.5864,0.8278,0.4541,0.5643,0.6303,0.8806,0.4908,0.5841,0.6059,0.8587,0.4681,0.5757


In [8]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

-rw-r--r-- 1 root root 1.6M Nov 11 03:46 results_small.zip


## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the row in `EXPERIMENTS.md`.